# Full-corpus main-topic classification: Starling-7B

# Load Packages & Set Working Directory

## Setup

In [ ]:
# packages

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import csv
import json
import re
import simpledorff
import pandas as pd
import transformers
from transformers import AutoTokenizer
from transformers import  LlamaForCausalLM, LlamaTokenizer, pipeline
import transformers

import torch
from torch import cuda, bfloat16, manual_seed

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.llms import HuggingFacePipeline

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score
from sklearn.metrics import classification_report

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
torch.clear_autocast_cache()

In [ ]:
os.getcwd()

# go one level up in the directory

huggingface_cache_dir = 'model_starling'

# change huggingface cache
os.environ['TRANSFORMERS_CACHE'] = huggingface_cache_dir

# Load the Model from Huggingface

In [ ]:
torch.manual_seed(0)

model_id = 'berkeley-nest/Starling-LM-7B-alpha'

device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

# begin initializing HF items, need auth token for these
model_config = transformers.AutoConfig.from_pretrained(
    model_id,
    cache_dir=huggingface_cache_dir
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    config=model_config,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=huggingface_cache_dir
)
model.eval()

print(f"Model loaded on {device}")

tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_id,
    cache_dir=huggingface_cache_dir)

In [ ]:
device = torch.device('cuda')
print("GPU Name:", torch.cuda.get_device_name(device))
print("Memory Usage:", torch.cuda.memory_allocated(device) / 1024 ** 3, "GB")
print("Max Memory Usage:", torch.cuda.max_memory_allocated(device) / 1024 ** 3, "GB")

In [ ]:
torch.manual_seed(0)
generate_text = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    pad_token_id=tokenizer.eos_token_id,
    temperature=0.0,
    max_new_tokens=512, 
    repetition_penalty=1.1  
)

llm = HuggingFacePipeline(pipeline=generate_text)

## Helper functions

In [ ]:
def regex_extract_to_dataframe(strings):
    # Initialize empty lists to store extracted values
    article_ids = []
    about_covid_values = []

    # Define regex pattern for article_id and about_covid with optional double quotes
    article_id_pattern = r'"article_id"\s*:\s*"?(\d+)"?'
    about_covid_pattern = r'"about_covid"\s*:\s*"?(\d)"?'

    # Iterate through each string
    for string_data in strings:
        # Use regex to find matches for article_id
        article_id_match = re.search(article_id_pattern, string_data)

        # Use regex to find matches for about_covid
        about_covid_match = re.search(about_covid_pattern, string_data)

        # Extract values from the regex matches
        article_id = int(article_id_match.group(1)) if article_id_match else None
        about_covid = int(about_covid_match.group(1)) if about_covid_match else None

        # Append values to the respective lists
        article_ids.append(article_id)
        about_covid_values.append(about_covid)

    # Create a DataFrame using the extracted values
    df = pd.DataFrame({
        "article_id": article_ids,
        "about_covid": about_covid_values
    })

    return df

In [ ]:
def zero_shot_prompt_messages(system_prompt, input_prompt, main_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input_prompt + "\n" + main_prompt},
    ]
    prompt = generate_text.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

In [ ]:
def analyze_dataframe(df, chain):
    generated_text_results = []
    torch.manual_seed(0)

    for index, row in df.iterrows():  
        article_id = row['article_id']
        text = row['Text']
        category = row['Category']
        keywords = row['Keywords']

        input_variables = {
            "article_id": article_id,
            "text": text,
            "category": category,
            "keywords": keywords
        }
        
        # Generate text using the chain
        generated_text = chain.run(input_variables)
        print(generated_text)
        generated_text_results.append(generated_text)
    
    return generated_text_results

# Data Prep

## Get Annotated NOS Articles DF

In [ ]:
df = pd.read_csv('data/final_nosarticles.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
df['page_id'] = df['page_id'].astype(int)

# remove line break
df['Text'] = df['Text'].str.replace('[LINE_BREAK]', '\n ')

print(df.shape)

In [ ]:
# are there duplicate articles? 
duplicates = df[df.duplicated(subset=['page_id'], keep=False)]

In [ ]:
df.Text.values[0]

In [ ]:
# create dataframe with only page_id, Text, Category, Keywords
df_analysis = df[['page_id', 'Text', 'Category', 'Keywords']]

# rename page_id to article_id
df_analysis.rename(columns={'page_id': 'article_id'}, inplace=True)

In [ ]:
# Shuffle the DataFrame for sampling different dates for each time
df_shuffled = df_analysis.sample(frac=1, random_state=42)

In [ ]:
dfs = []
chunk_size = 500

for i in range(0, len(df_shuffled), chunk_size):
    dfs.append(df_shuffled.iloc[i:i+chunk_size])

# Access the smaller DataFrames
for i, smaller_df in enumerate(dfs):
    print(f"DataFrame {i+1} has {len(smaller_df)} rows")

In [ ]:
# Naming the smaller DataFrames dynamically
for i, smaller_df in enumerate(dfs):
    globals()[f'df_{i+1}'] = smaller_df

In [ ]:
for i in range(len(dfs)):
    print(f"DataFrame df_{i+1} has {len(globals()[f'df_{i+1}'])} rows")

## Prompt building

In [ ]:
system_prompt = """
As a helpful AI assistant, your task is to determine the main topic of news articles. Articles may focus on either the "Coronavirus and/or the COVID-19 pandemic" or some other topic.
A main topic is the overarching theme discussed in the majority of the news article. For an article to have the main topic of the "Coronavirus and/or the COVID-19 pandemic", it should predominantly discuss these subjects.
"""

input_prompt = """
Read the following article with the ID {article_id}: {text} \n
This article falls under the categories: {category} and contains the keywords: {keywords}.
"""

main_prompt = """
Take a moment to understand the article. 
Remember, for a topic to be a main topic of the news article, it should be discussed in the majority of the article. 

Based on the information provided, determine if the main topic of this news article is the "Coronavirus and/or the COVID-19 pandemic" or another subject. 
Assign a value of 1 if the main topic is the "Coronavirus and/or the COVID-19 pandemic", and a value of 0 if it is another subject.

Output your results in JSON format with keys "article_id" and "about_covid", where the article ID and your answer are the values. 
Follow the example output format provided. Do not include any additional information or explanation. 

Example Output (JSON format):
{{
    "article_id": "2000000",
    "about_covid": "1"
}}
"""

# About Covid

## Zero-Shot Classifier About_Covid

In [ ]:
zero_shot_prompt = zero_shot_prompt_messages(system_prompt, input_prompt, main_prompt)
print(zero_shot_prompt)

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["article_id", "text", "category", "keywords"],
    template=zero_shot_prompt
)

In [ ]:
chain = LLMChain(llm = llm, prompt = prompt_template, output_key="article_id, about_covid")

In [ ]:
%%time
# analyze the first dataframe
# df_1_results_list = analyze_dataframe(df_1, chain)
# df_2_results_list = analyze_dataframe(df_2, chain)
# df_3_results_list = analyze_dataframe(df_3, chain)
df_4_results_list = analyze_dataframe(df_4, chain)
# df_5_results_list = analyze_dataframe(df_5, chain)
# df_6_results_list = analyze_dataframe(df_6, chain)
# df_7_results_list = analyze_dataframe(df_7, chain)
# df_8_results_list = analyze_dataframe(df_8, chain)
# df_9_results_list = analyze_dataframe(df_9, chain)
# df_10_results_list = analyze_dataframe(df_10, chain)
# df_11_results_list = analyze_dataframe(df_11, chain)
# df_12_results_list = analyze_dataframe(df_12, chain)
# df_13_results_list = analyze_dataframe(df_13, chain)
# df_14_results_list = analyze_dataframe(df_14, chain)
# df_15_results_list = analyze_dataframe(df_15, chain)
# df_16_results_list = analyze_dataframe(df_16, chain)
# df_17_results_list = analyze_dataframe(df_17, chain)
# df_18_results_list = analyze_dataframe(df_18, chain)
# df_19_results_list = analyze_dataframe(df_19, chain)
# df_20_results_list = analyze_dataframe(df_20, chain)
# df_21_results_list = analyze_dataframe(df_21, chain)
# df_22_results_list = analyze_dataframe(df_22, chain)
# df_23_results_list = analyze_dataframe(df_23, chain)
# df_24_results_list = analyze_dataframe(df_24, chain)
# df_25_results_list = analyze_dataframe(df_25, chain)
# df_26_results_list = analyze_dataframe(df_26, chain)

In [ ]:
# df_1_results = regex_extract_to_dataframe(df_1_results_list)
# df_2_results = regex_extract_to_dataframe(df_2_results_list)
# df_3_results = regex_extract_to_dataframe(df_3_results_list)
df_4_results = regex_extract_to_dataframe(df_4_results_list)
# df_5_results = regex_extract_to_dataframe(df_5_results_list)
# df_6_results = regex_extract_to_dataframe(df_6_results_list)
# df_7_results = regex_extract_to_dataframe(df_7_results_list)
# df_8_results = regex_extract_to_dataframe(df_8_results_list)
# df_9_results = regex_extract_to_dataframe(df_9_results_list)
# df_10_results = regex_extract_to_dataframe(df_10_results_list)
# df_11_results = regex_extract_to_dataframe(df_11_results_list)
# df_12_results = regex_extract_to_dataframe(df_12_results_list)
# df_13_results = regex_extract_to_dataframe(df_13_results_list)
# df_14_results = regex_extract_to_dataframe(df_14_results_list)
# df_15_results = regex_extract_to_dataframe(df_15_results_list)
# df_16_results = regex_extract_to_dataframe(df_16_results_list)
# df_17_results = regex_extract_to_dataframe(df_17_results_list)
# df_18_results = regex_extract_to_dataframe(df_18_results_list)
# df_19_results = regex_extract_to_dataframe(df_19_results_list)
# df_20_results = regex_extract_to_dataframe(df_20_results_list)
# df_21_results = regex_extract_to_dataframe(df_21_results_list)
# df_22_results = regex_extract_to_dataframe(df_22_results_list)
# df_23_results = regex_extract_to_dataframe(df_23_results_list)
# df_24_results = regex_extract_to_dataframe(df_24_results_list)
# df_25_results = regex_extract_to_dataframe(df_25_results_list)
# df_26_results = regex_extract_to_dataframe(df_26_results_list)

In [ ]:
# # merge df with results
# df_1_merged = pd.merge(df_1, df_1_results, on='article_id', how='left')
# df_2_merged = pd.merge(df_2, df_2_results, on='article_id', how='left')
# df_3_merged = pd.merge(df_3, df_3_results, on='article_id', how='left')
df_4_merged = pd.merge(df_4, df_4_results, on='article_id', how='left')
# df_5_merged = pd.merge(df_5, df_5_results, on='article_id', how='left')
# df_6_merged = pd.merge(df_6, df_6_results, on='article_id', how='left')
# df_7_merged = pd.merge(df_7, df_7_results, on='article_id', how='left')
# df_8_merged = pd.merge(df_8, df_8_results, on='article_id', how='left')
# df_9_merged = pd.merge(df_9, df_9_results, on='article_id', how='left')
# df_10_merged = pd.merge(df_10, df_10_results, on='article_id', how='left')
# df_11_merged = pd.merge(df_11, df_11_results, on='article_id', how='left')
# df_12_merged = pd.merge(df_12, df_12_results, on='article_id', how='left')
# df_13_merged = pd.merge(df_13, df_13_results, on='article_id', how='left')
# df_14_merged = pd.merge(df_14, df_14_results, on='article_id', how='left')
# df_15_merged = pd.merge(df_15, df_15_results, on='article_id', how='left')
# df_16_merged = pd.merge(df_16, df_16_results, on='article_id', how='left')
# df_17_merged = pd.merge(df_17, df_17_results, on='article_id', how='left')
# df_18_merged = pd.merge(df_18, df_18_results, on='article_id', how='left')
# df_19_merged = pd.merge(df_19, df_19_results, on='article_id', how='left')
# df_20_merged = pd.merge(df_20, df_20_results, on='article_id', how='left')
# df_21_merged = pd.merge(df_21, df_21_results, on='article_id', how='left')
# df_22_merged = pd.merge(df_22, df_22_results, on='article_id', how='left')
# df_23_merged = pd.merge(df_23, df_23_results, on='article_id', how='left')
# df_24_merged = pd.merge(df_24, df_24_results, on='article_id', how='left')
# df_25_merged = pd.merge(df_25, df_25_results, on='article_id', how='left')
# df_26_merged = pd.merge(df_26, df_26_results, on='article_id', how='left')

In [ ]:
# combine all dataframes
df_coded = pd.concat([df_1_merged, df_2_merged, df_3_merged, df_4_merged, df_5_merged, 
df_6_merged, df_7_merged, df_8_merged, df_9_merged, df_10_merged, df_11_merged, df_12_merged, 
df_13_merged, df_14_merged, df_15_merged, df_16_merged, df_17_merged, df_18_merged, df_19_merged,
df_20_merged, df_21_merged, df_22_merged, df_23_merged, df_24_merged, df_25_merged, df_26_merged])

In [ ]:
df_coded['Text'] = df_coded['Text'].str.replace('\n ', '[LINE_BREAK]')

In [ ]:
df_coded['Text'].values[0]

In [ ]:
df_coded.to_csv('data/about_covid_allcoded_NOS.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)